### Implementing Logistic Regression using Gradient Descent and comparing it with scikit learn's implementation on a toy dataset

the loss function which in gradient descent is derived from the concept of maximum likelyhood.      
- rough estimate of our loss function used- sum(log (max likelyhood))

In [62]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score , precision_score , recall_score,f1_score

In [63]:
df = pd.read_csv('placement.csv')

In [64]:
df.head()

,Unnamed: 0,cgpa,iq,placement
0,0,6.8,123.0,1
1,1,5.9,106.0,0
2,2,5.3,121.0,0
3,3,7.4,132.0,1
4,4,5.8,142.0,0


In [65]:
df=df.drop(columns=['Unnamed: 0'])
df.head()

,cgpa,iq,placement
0,6.8,123.0,1
1,5.9,106.0,0
2,5.3,121.0,0
3,7.4,132.0,1
4,5.8,142.0,0


In [66]:
X= df.iloc[:,0:2]
y=df.iloc[:,-1]
X,y

(    cgpa     iq
 0    6.8  123.0
 1    5.9  106.0
 2    5.3  121.0
 3    7.4  132.0
 4    5.8  142.0
 ..   ...    ...
 95   4.3  200.0
 96   4.4   42.0
 97   6.7  182.0
 98   6.3  103.0
 99   6.2  113.0
 
 [100 rows x 2 columns],
 0     1
 1     0
 2     0
 3     1
 4     0
      ..
 95    0
 96    0
 97    1
 98    1
 99    1
 Name: placement, Length: 100, dtype: int64)

In [67]:
def sigmoid(z):
    return 1/(1+np.exp(-z))

In [68]:
class GradientLogisticRegression:
    def __init__(self,lr=0.01,epochs=1000):
        self.epochs= epochs 
        self.learning_rate=lr
        self.weights=None
    def fit(self,X_train,y_train):
        X_train=np.insert(X_train,0,1,axis=1)
        self.weights= np.ones(X_train.shape[1])
        for i in range(self.epochs):
            y_hat=sigmoid(np.dot(X_train,self.weights))
            self.weights=self.weights + self.learning_rate * (np.dot((y_train - y_hat), X_train) / X_train.shape[0])
        return self.weights
    def predict(self, X):
        X = np.insert(X, 0, 1, axis=1)
        return (sigmoid(np.dot(X, self.weights)) >= 0.5).astype(int)

In [69]:
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=42)

In [70]:
gr= GradientLogisticRegression()
gr.fit(X_train,y_train)

array([ 0.94890261,  5.16285514, -0.69719997])

In [71]:
y_pred=gr.predict(X_test)

In [72]:
print(f"Our implementation without feature scaling : {accuracy_score(y_test,y_pred)}")

Our implementation without feature scaling : 0.6


In [73]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

gr = GradientLogisticRegression()
gr.fit(X_train_scaled, y_train)
y_pred = gr.predict(X_test_scaled)

print(f"Our implementation with feature scaling : {accuracy_score(y_test, y_pred)}")

Our implementation with feature scaling : 0.85


In [74]:
lr=LogisticRegression()
lr.fit(X_train,y_train)
y_pred_sklearn=lr.predict(X_test)

In [75]:
print(f"accuracy of logistic regression without feature scaling : {accuracy_score(y_test,y_pred_sklearn)}")

accuracy of logistic regression without feature scaling : 0.85


In [76]:
lr.fit(X_train_scaled, y_train)          
y_pred_sklearn = lr.predict(X_test_scaled)
print(f"accuracy of logistic regression with feature scaling : {accuracy_score(y_test, y_pred_sklearn)}")

accuracy of logistic regression with feature scaling : 0.85


## Experimental Results & Analysis (placement.csv)
 
| Model | Feature Scaling | Accuracy |
|---|---|---|
| Custom GD |  No | **0.60** |
| Custom GD |  Yes | **0.85** |
| Sklearn LR |  No | **0.85** |
| Sklearn LR |  Yes | **0.85** |
 


### Why did our custom GD score only 0.60 without scaling?
 
This is a textbook demonstration of **why feature scaling is non-negotiable for gradient descent**.
 
The `placement.csv` dataset has features on **very different scales** (e.g., CGPA in range 6–10 vs. IQ scores in range 100–160 or package in LPA). This causes:
 
1. **Elongated loss surface** — the gradient is dominated by the large-scale feature, causing the optimizer to take steps that overshoot in one dimension and undershoot in another.
2. **Oscillating / diverging updates** — the weight corresponding to the large-scale feature gets huge gradient updates while the small-scale feature's weight barely moves.
3. **Premature convergence** — with a fixed learning rate (`lr=0.01`) and fixed epochs (`1000`), gradient descent gets stuck far from the true minimum before the budget runs out.


### Why did Sklearn score 0.85 even without scaling?
 
Sklearn's `LogisticRegression` uses the **L-BFGS** solver by default — a **quasi-Newton second-order optimizer** that approximates the Hessian (curvature of the loss). Because it accounts for curvature, it naturally adapts step sizes per dimension, making it far **more robust to unscaled features** than vanilla gradient descent.
 


In [77]:
X, y = make_classification(n_samples=500, n_features=2, n_informative=1,n_redundant=0,n_classes=2, n_clusters_per_class=1, random_state=42,hypercube=False,class_sep=1)
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=42)

In [78]:
gr_toy= GradientLogisticRegression()
gr_toy.fit(X_train,y_train)
y_pred=gr.predict(X_test)


In [79]:
lr_toy=LogisticRegression()
lr_toy.fit(X_train,y_train)
y_pred_sklearn=lr.predict(X_test)

In [80]:
print(f"our implementation : {accuracy_score(y_test, y_pred)}\nSklearn implementation : {accuracy_score(y_test, y_pred_sklearn)}")

our implementation : 0.74
Sklearn implementation : 0.75


### What are the assumptions of Logistic Regression
1. **Binary output** (for binary LR)
2. **No multicollinearity** among features
3. **Linear relationship** between features and the **log-odds**
4. **Large sample size** (MLE needs sufficient data)
5. **Independence** of observations
